# 04 BM25


**Nombre:** Harry Guajan  
**Fecha:** 13 de Noviembre de 2026



## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [2]:
# Reutilizado de 02_booleansearch y 03_vectorsearch: corpus Gutenberg,
# tokenización/normalización, títulos y las fórmulas TF-IDF/BM25.
from collections import Counter
import math
import re

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
    if len(identificadores) != 18:
        raise LookupError("La colección Gutenberg de NLTK no está disponible.")
except LookupError:
    nltk.download("gutenberg", quiet=True)
    from nltk.corpus import gutenberg
    identificadores = gutenberg.fileids()
if len(identificadores) != 18:
    raise RuntimeError(f"Se esperaban 18 documentos y se encontraron {len(identificadores)}.")
documentos = [gutenberg.raw(doc_id) for doc_id in identificadores]
token_pattern = re.compile(r"(?u)\b\w\w+\b")

def tokenizar(texto):
    return token_pattern.findall(texto.lower())

def titulo(doc_id):
    nombres = {
        "austen-emma.txt": "Emma", "austen-persuasion.txt": "Persuasion", "austen-sense.txt": "Sense and Sensibility",
        "bible-kjv.txt": "The King James Bible", "blake-poems.txt": "Poems of William Blake", "bryant-stories.txt": "Stories by William Cullen Bryant",
        "burgess-busterbrown.txt": "The Adventures of Buster Bear", "carroll-alice.txt": "Alice's Adventures in Wonderland",
        "chesterton-ball.txt": "The Ball and the Cross", "chesterton-brown.txt": "The Wisdom of Father Brown", "chesterton-thursday.txt": "The Man Who Was Thursday",
        "edgeworth-parents.txt": "The Parent's Assistant", "melville-moby_dick.txt": "Moby Dick", "milton-paradise.txt": "Paradise Lost",
        "shakespeare-caesar.txt": "The Tragedy of Julius Caesar", "shakespeare-hamlet.txt": "The Tragedy of Hamlet", "shakespeare-macbeth.txt": "The Tragedy of Macbeth", "whitman-leaves.txt": "Leaves of Grass",
    }
    return nombres.get(doc_id, doc_id)

documentos_tokenizados = [tokenizar(texto) for texto in documentos]
vocabulario = sorted({termino for tokens in documentos_tokenizados for termino in tokens})
indice_terminos = {termino: i for i, termino in enumerate(vocabulario)}
matriz_tf = np.zeros((len(documentos), len(vocabulario)), dtype=np.int64)
for fila, tokens in enumerate(documentos_tokenizados):
    for termino, frecuencia in Counter(tokens).items():
        matriz_tf[fila, indice_terminos[termino]] = frecuencia

df = np.count_nonzero(matriz_tf, axis=0)
idf = np.log(len(documentos) / df) + 1
longitudes = np.array([len(tokens) for tokens in documentos_tokenizados])
longitud_media = float(longitudes.mean())
tabla_corpus = pd.DataFrame({
    "Identificador de documento": identificadores,
    "Título": [titulo(doc_id) for doc_id in identificadores],
    "Texto": documentos,
    "tokens": documentos_tokenizados,
})
tabla_estadisticas = pd.DataFrame({"término": vocabulario, "df": df, "idf": idf})
display(pd.DataFrame([{"Documentos": len(documentos), "Vocabulario": len(vocabulario),
                       "Longitud media": longitud_media}]))
display(tabla_corpus[["Identificador de documento", "Título", "Texto", "tokens"]].head())
display(tabla_estadisticas.sort_values(["df", "término"], ascending=[False, True]).head(15))

# La representación de documentos se conserva dispersa, como pide el enunciado.
vectorizador = TfidfVectorizer(
    tokenizer=lambda tokens: tokens, preprocessor=lambda tokens: tokens,
    token_pattern=None, lowercase=False, use_idf=True, smooth_idf=False,
    sublinear_tf=False, norm=None,
)
matriz_tfidf = vectorizador.fit_transform(documentos_tokenizados)
if not np.array_equal(vectorizador.get_feature_names_out(), np.array(vocabulario)):
    raise AssertionError("El vocabulario disperso no coincide con el vocabulario manual.")

def puntuaciones_bm25(consulta_tokens, k1=1.2, b=0.75):
    puntuaciones = np.zeros(len(documentos), dtype=float)
    consulta_unica = set(consulta_tokens)
    for i, tokens in enumerate(documentos_tokenizados):
        frecuencias = Counter(tokens)
        ajuste_longitud = 1 - b + b * len(tokens) / longitud_media
        for termino in consulta_unica:
            if termino not in indice_terminos:
                continue
            frecuencia_documental = df[indice_terminos[termino]]
            idf_bm25 = math.log((len(documentos) - frecuencia_documental + 0.5) / (frecuencia_documental + 0.5) + 1)
            frecuencia = frecuencias[termino]
            puntuaciones[i] += idf_bm25 * frecuencia * (k1 + 1) / (frecuencia + k1 * ajuste_longitud)
    return puntuaciones


,Identificador de documento,Título,Texto,tokens
0,austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAP...,"[emma, by, jane, austen, 1816, volume, chapter..."
1,austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter ...,"[persuasion, by, jane, austen, 1818, chapter, ..."
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\...,"[sense, and, sensibility, by, jane, austen, 18..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of...,"[the, king, james, bible, the, old, testament,..."
4,blake-poems.txt,Poems of William Blake,[Poems by William Blake 1789]\n\n \nSONGS OF I...,"[poems, by, william, blake, 1789, songs, of, i..."


,término,df,idf
704,about,18,1.0
1314,after,18,1.0
1327,against,18,1.0
1335,age,18,1.0
1600,all,18,1.0
1671,alone,18,1.0
1724,am,18,1.0
1833,among,18,1.0
1879,an,18,1.0
1938,and,18,1.0


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [3]:
consultas = {
    "frecuente corta": ("man", "encuentra referencias generales a personas"),
    "frecuente media": ("love time", "busca relaciones entre amor y tiempo"),
    "raro corto": ("whale", "busca referencias explícitas a ballenas"),
    "raro media": ("ship whale ocean", "busca escenas marítimas y navegación"),
    "larga narrativa": ("young man went house", "busca una escena narrativa con sujeto, acción y lugar"),
    "larga literaria": ("king queen night word", "busca lenguaje de poder y ambiente nocturno"),
    "temática": ("church family father", "busca contexto religioso y familiar"),
    "mixta": ("science machine life", "busca una combinación científica y existencial"),
}
def ordenar_resultados(scores):
    positivos = [(identificadores[i], titulo(identificadores[i]), float(scores[i]))
                 for i in range(len(identificadores)) if scores[i] > 0]
    return sorted(positivos, key=lambda fila: (-fila[2], str(fila[0])))[:10]

resultados = {}
for nombre, (texto_consulta, objetivo) in consultas.items():
    tokens_consulta = tokenizar(texto_consulta)
    vector_consulta = vectorizador.transform([tokens_consulta])
    scores_tfidf = (matriz_tfidf @ vector_consulta.T).toarray().ravel()
    norma_consulta = float(np.sqrt(vector_consulta.multiply(vector_consulta).sum()))
    normas_documentos = np.sqrt(matriz_tfidf.multiply(matriz_tfidf).sum(axis=1)).A1
    denominadores = normas_documentos * norma_consulta
    scores_coseno = np.divide(scores_tfidf, denominadores, out=np.zeros_like(scores_tfidf), where=denominadores != 0)
    scores_bm25 = puntuaciones_bm25(tokens_consulta)
    resultados[nombre] = {"texto": texto_consulta, "objetivo": objetivo, "tokens": tokens_consulta,
                         "tfidf": scores_coseno, "bm25": scores_bm25}
    display(pd.DataFrame([(t, int(df[indice_terminos[t]])) for t in sorted(set(tokens_consulta)) if t in indice_terminos],
                         columns=["término", "df"]))
    tabla_tfidf = pd.DataFrame(ordenar_resultados(scores_coseno), columns=["doc_id", "Título", "Coseno"])
    tabla_tfidf.insert(0, "consulta", nombre)
    tabla_tfidf.insert(1, "objetivo", objetivo)
    display(tabla_tfidf)
    tabla_bm25 = pd.DataFrame(ordenar_resultados(scores_bm25), columns=["doc_id", "Título", "Score BM25"])
    tabla_bm25.insert(0, "consulta", nombre)
    tabla_bm25.insert(1, "objetivo", objetivo)
    display(tabla_bm25)


,doc_id,Título,Coseno
0,chesterton-thursday.txt,The Man Who Was Thursday,0.047094
1,chesterton-brown.txt,The Wisdom of Father Brown,0.044329
2,chesterton-ball.txt,The Ball and the Cross,0.041407
3,shakespeare-caesar.txt,The Tragedy of Julius Caesar,0.029499
4,bible-kjv.txt,The King James Bible,0.026343
5,milton-paradise.txt,Paradise Lost,0.026239
6,melville-moby_dick.txt,Moby Dick,0.025305
7,bryant-stories.txt,Stories by William Cullen Bryant,0.021224
8,shakespeare-macbeth.txt,The Tragedy of Macbeth,0.021056
9,whitman-leaves.txt,Leaves of Grass,0.018037


,doc_id,Título,Score BM25
0,bible-kjv.txt,The King James Bible,0.058529
1,chesterton-thursday.txt,The Man Who Was Thursday,0.058515
2,chesterton-ball.txt,The Ball and the Cross,0.058512
3,chesterton-brown.txt,The Wisdom of Father Brown,0.058511
4,melville-moby_dick.txt,Moby Dick,0.058457
5,whitman-leaves.txt,Leaves of Grass,0.058376
6,milton-paradise.txt,Paradise Lost,0.058368
7,edgeworth-parents.txt,The Parent's Assistant,0.058326
8,bryant-stories.txt,Stories by William Cullen Bryant,0.058318
9,austen-emma.txt,Emma,0.058299


,doc_id,Título,Coseno
0,blake-poems.txt,Poems of William Blake,0.042531
1,whitman-leaves.txt,Leaves of Grass,0.026672
2,austen-sense.txt,Sense and Sensibility,0.020097
3,austen-emma.txt,Emma,0.019538
4,edgeworth-parents.txt,The Parent's Assistant,0.018941
5,shakespeare-macbeth.txt,The Tragedy of Macbeth,0.018397
6,milton-paradise.txt,Paradise Lost,0.017944
7,burgess-busterbrown.txt,The Adventures of Buster Bear,0.017778
8,austen-persuasion.txt,Persuasion,0.017425
9,carroll-alice.txt,Alice's Adventures in Wonderland,0.016963


,doc_id,Título,Score BM25
0,whitman-leaves.txt,Leaves of Grass,0.504191
1,milton-paradise.txt,Paradise Lost,0.501428
2,austen-emma.txt,Emma,0.500620
3,austen-sense.txt,Sense and Sensibility,0.499444
4,edgeworth-parents.txt,The Parent's Assistant,0.498101
5,blake-poems.txt,Poems of William Blake,0.497385
6,bible-kjv.txt,The King James Bible,0.496691
7,austen-persuasion.txt,Persuasion,0.496572
8,chesterton-ball.txt,The Ball and the Cross,0.490230
9,bryant-stories.txt,Stories by William Cullen Bryant,0.484983


,doc_id,Título,Coseno
0,melville-moby_dick.txt,Moby Dick,0.123545
1,bryant-stories.txt,Stories by William Cullen Bryant,0.006660
2,shakespeare-hamlet.txt,The Tragedy of Hamlet,0.001593
3,whitman-leaves.txt,Leaves of Grass,0.000779
4,chesterton-ball.txt,The Ball and the Cross,0.000514
5,bible-kjv.txt,The King James Bible,0.000061


,doc_id,Título,Score BM25
0,melville-moby_dick.txt,Moby Dick,2.356112
1,bryant-stories.txt,Stories by William Cullen Bryant,2.268169
2,whitman-leaves.txt,Leaves of Grass,1.896076
3,shakespeare-hamlet.txt,The Tragedy of Hamlet,1.871798
4,chesterton-ball.txt,The Ball and the Cross,1.621240
5,bible-kjv.txt,The King James Bible,0.738836


... (6 outputs left out)

,doc_id,Título,Coseno
0,chesterton-brown.txt,The Wisdom of Father Brown,0.024556
1,blake-poems.txt,Poems of William Blake,0.023323
2,austen-persuasion.txt,Persuasion,0.016963
3,edgeworth-parents.txt,The Parent's Assistant,0.012693
4,austen-emma.txt,Emma,0.012250
5,bryant-stories.txt,Stories by William Cullen Bryant,0.010732
6,shakespeare-hamlet.txt,The Tragedy of Hamlet,0.010628
7,austen-sense.txt,Sense and Sensibility,0.008108
8,bible-kjv.txt,The King James Bible,0.007296
9,chesterton-ball.txt,The Ball and the Cross,0.006154


,doc_id,Título,Score BM25
0,chesterton-brown.txt,The Wisdom of Father Brown,1.474927
1,bible-kjv.txt,The King James Bible,1.441346
2,bryant-stories.txt,Stories by William Cullen Bryant,1.426244
3,austen-persuasion.txt,Persuasion,1.424482
4,austen-sense.txt,Sense and Sensibility,1.423008
5,austen-emma.txt,Emma,1.407071
6,edgeworth-parents.txt,The Parent's Assistant,1.401308
7,chesterton-ball.txt,The Ball and the Cross,1.392694
8,chesterton-thursday.txt,The Man Who Was Thursday,1.343901
9,melville-moby_dick.txt,Moby Dick,1.340008


,doc_id,Título,Coseno
0,chesterton-ball.txt,The Ball and the Cross,0.008066
1,whitman-leaves.txt,Leaves of Grass,0.007998
2,chesterton-brown.txt,The Wisdom of Father Brown,0.007978
3,shakespeare-hamlet.txt,The Tragedy of Hamlet,0.004708
4,milton-paradise.txt,Paradise Lost,0.004468
5,shakespeare-macbeth.txt,The Tragedy of Macbeth,0.004022
6,blake-poems.txt,Poems of William Blake,0.003757
7,melville-moby_dick.txt,Moby Dick,0.003101
8,chesterton-thursday.txt,The Man Who Was Thursday,0.002223
9,austen-persuasion.txt,Persuasion,0.002038


,doc_id,Título,Score BM25
0,chesterton-brown.txt,The Wisdom of Father Brown,4.337437
1,chesterton-ball.txt,The Ball and the Cross,4.334890
2,whitman-leaves.txt,Leaves of Grass,3.632174
3,melville-moby_dick.txt,Moby Dick,2.332821
4,shakespeare-hamlet.txt,The Tragedy of Hamlet,1.850363
5,chesterton-thursday.txt,The Man Who Was Thursday,1.553381
6,milton-paradise.txt,Paradise Lost,0.984556
7,edgeworth-parents.txt,The Parent's Assistant,0.751587
8,bible-kjv.txt,The King James Bible,0.470235
9,austen-emma.txt,Emma,0.057593


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

### Comparación y aportes por término

Se comparan los rankings de ambos métodos y se muestra el aporte de cada término de la consulta explicada.

In [6]:
def tabla_comparacion(nombre):
    datos = resultados[nombre]
    ranking_tfidf = ordenar_resultados(datos["tfidf"])
    ranking_bm25 = ordenar_resultados(datos["bm25"])
    filas = []
    for posicion in range(max(len(ranking_tfidf), len(ranking_bm25))):
        tfidf = ranking_tfidf[posicion] if posicion < len(ranking_tfidf) else ("", "", np.nan)
        bm25 = ranking_bm25[posicion] if posicion < len(ranking_bm25) else ("", "", np.nan)
        filas.append({"Posición": posicion + 1, "ID y título TF-IDF": f"{tfidf[0]} — {tfidf[1]}" if tfidf[0] else "",
                      "Coseno": tfidf[2], "ID y título BM25": f"{bm25[0]} — {bm25[1]}" if bm25[0] else "",
                      "Score BM25": bm25[2]})
    ids_tfidf = {fila[0] for fila in ranking_tfidf}
    ids_bm25 = {fila[0] for fila in ranking_bm25}
    resumen = {"consulta": nombre, "tamaño TF-IDF": len(ranking_tfidf), "tamaño BM25": len(ranking_bm25),
               "intersección top 10": len(ids_tfidf & ids_bm25),
               "primer TF-IDF": ranking_tfidf[0][0] if ranking_tfidf else "",
               "primer BM25": ranking_bm25[0][0] if ranking_bm25 else ""}
    return pd.DataFrame(filas), resumen

resumenes = []
for nombre in consultas:
    tabla, resumen = tabla_comparacion(nombre)
    resumenes.append(resumen)
    tabla.insert(0, "consulta", nombre)
    display(tabla)
display(pd.DataFrame(resumenes))

consulta_explicada = "raro media"
ranking_a = ordenar_resultados(resultados[consulta_explicada]["tfidf"])
ranking_b = ordenar_resultados(resultados[consulta_explicada]["bm25"])
ids_a = [fila[0] for fila in ranking_a]
ids_b = [fila[0] for fila in ranking_b]
candidatos = [doc_id for doc_id in ids_a if doc_id in ids_b and ids_a.index(doc_id) != ids_b.index(doc_id)]
documentos_explicados = candidatos[:2] if len(candidatos) >= 2 else list(dict.fromkeys(ids_a[:2] + ids_b[:2]))[:2]
filas_aportes = []
for doc_id in documentos_explicados:
    i = identificadores.index(doc_id)
    frecuencias = Counter(documentos_tokenizados[i])
    for termino in sorted(set(resultados[consulta_explicada]["tokens"])):
        if termino not in indice_terminos:
            continue
        f_td = frecuencias[termino]
        df_t = int(df[indice_terminos[termino]])
        ajuste = 1 - 0.75 + 0.75 * len(documentos_tokenizados[i]) / longitud_media
        idf_bm = math.log((len(documentos) - df_t + 0.5) / (df_t + 0.5) + 1)
        aporte = idf_bm * f_td * 2.2 / (f_td + 1.2 * ajuste)
        filas_aportes.append({"doc_id": doc_id, "|d|": len(documentos_tokenizados[i]), "término": termino,
                              "f(t,d)": f_td, "df(t)": df_t, "aporte BM25": aporte,
                              "posición TF-IDF": ids_a.index(doc_id) + 1 if doc_id in ids_a else None,
                              "posición BM25": ids_b.index(doc_id) + 1 if doc_id in ids_b else None})
display(pd.DataFrame(filas_aportes))


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,chesterton-thursday.txt — The Man Who Was Thur...,0.047094,bible-kjv.txt — The King James Bible,0.058529
1,2,chesterton-brown.txt — The Wisdom of Father Brown,0.044329,chesterton-thursday.txt — The Man Who Was Thur...,0.058515
2,3,chesterton-ball.txt — The Ball and the Cross,0.041407,chesterton-ball.txt — The Ball and the Cross,0.058512
3,4,shakespeare-caesar.txt — The Tragedy of Julius...,0.029499,chesterton-brown.txt — The Wisdom of Father Brown,0.058511
4,5,bible-kjv.txt — The King James Bible,0.026343,melville-moby_dick.txt — Moby Dick,0.058457
5,6,milton-paradise.txt — Paradise Lost,0.026239,whitman-leaves.txt — Leaves of Grass,0.058376
6,7,melville-moby_dick.txt — Moby Dick,0.025305,milton-paradise.txt — Paradise Lost,0.058368
7,8,bryant-stories.txt — Stories by William Cullen...,0.021224,edgeworth-parents.txt — The Parent's Assistant,0.058326
8,9,shakespeare-macbeth.txt — The Tragedy of Macbeth,0.021056,bryant-stories.txt — Stories by William Cullen...,0.058318
9,10,whitman-leaves.txt — Leaves of Grass,0.018037,austen-emma.txt — Emma,0.058299


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,blake-poems.txt — Poems of William Blake,0.042531,whitman-leaves.txt — Leaves of Grass,0.504191
1,2,whitman-leaves.txt — Leaves of Grass,0.026672,milton-paradise.txt — Paradise Lost,0.501428
2,3,austen-sense.txt — Sense and Sensibility,0.020097,austen-emma.txt — Emma,0.500620
3,4,austen-emma.txt — Emma,0.019538,austen-sense.txt — Sense and Sensibility,0.499444
4,5,edgeworth-parents.txt — The Parent's Assistant,0.018941,edgeworth-parents.txt — The Parent's Assistant,0.498101
5,6,shakespeare-macbeth.txt — The Tragedy of Macbeth,0.018397,blake-poems.txt — Poems of William Blake,0.497385
6,7,milton-paradise.txt — Paradise Lost,0.017944,bible-kjv.txt — The King James Bible,0.496691
7,8,burgess-busterbrown.txt — The Adventures of Bu...,0.017778,austen-persuasion.txt — Persuasion,0.496572
8,9,austen-persuasion.txt — Persuasion,0.017425,chesterton-ball.txt — The Ball and the Cross,0.490230
9,10,carroll-alice.txt — Alice's Adventures in Wond...,0.016963,bryant-stories.txt — Stories by William Cullen...,0.484983


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt — Moby Dick,0.123545,melville-moby_dick.txt — Moby Dick,2.356112
1,2,bryant-stories.txt — Stories by William Cullen...,0.006660,bryant-stories.txt — Stories by William Cullen...,2.268169
2,3,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.001593,whitman-leaves.txt — Leaves of Grass,1.896076
3,4,whitman-leaves.txt — Leaves of Grass,0.000779,shakespeare-hamlet.txt — The Tragedy of Hamlet,1.871798
4,5,chesterton-ball.txt — The Ball and the Cross,0.000514,chesterton-ball.txt — The Ball and the Cross,1.621240
5,6,bible-kjv.txt — The King James Bible,0.000061,bible-kjv.txt — The King James Bible,0.738836


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt — Moby Dick,0.105090,melville-moby_dick.txt — Moby Dick,4.763493
1,2,whitman-leaves.txt — Leaves of Grass,0.006866,bryant-stories.txt — Stories by William Cullen...,4.396661
2,3,bryant-stories.txt — Stories by William Cullen...,0.005989,whitman-leaves.txt — Leaves of Grass,4.278024
3,4,chesterton-ball.txt — The Ball and the Cross,0.003610,shakespeare-hamlet.txt — The Tragedy of Hamlet,3.604763
4,5,milton-paradise.txt — Paradise Lost,0.002346,chesterton-ball.txt — The Ball and the Cross,3.568492
5,6,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.001950,milton-paradise.txt — Paradise Lost,1.925129
6,7,chesterton-brown.txt — The Wisdom of Father Brown,0.001819,chesterton-brown.txt — The Wisdom of Father Brown,1.711157
7,8,shakespeare-macbeth.txt — The Tragedy of Macbeth,0.000986,shakespeare-macbeth.txt — The Tragedy of Macbeth,1.706764
8,9,austen-persuasion.txt — Persuasion,0.000499,bible-kjv.txt — The King James Bible,1.581850
9,10,bible-kjv.txt — The King James Bible,0.000486,shakespeare-caesar.txt — The Tragedy of Julius...,1.050327


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,chesterton-brown.txt — The Wisdom of Father Brown,0.035981,bible-kjv.txt — The King James Bible,0.232815
1,2,chesterton-thursday.txt — The Man Who Was Thur...,0.032420,edgeworth-parents.txt — The Parent's Assistant,0.232593
2,3,chesterton-ball.txt — The Ball and the Cross,0.031423,chesterton-brown.txt — The Wisdom of Father Brown,0.232107
3,4,bible-kjv.txt — The King James Bible,0.031106,austen-sense.txt — Sense and Sensibility,0.231602
4,5,bryant-stories.txt — Stories by William Cullen...,0.029258,austen-persuasion.txt — Persuasion,0.231310
5,6,edgeworth-parents.txt — The Parent's Assistant,0.024848,chesterton-ball.txt — The Ball and the Cross,0.231273
6,7,austen-persuasion.txt — Persuasion,0.021898,austen-emma.txt — Emma,0.231098
7,8,shakespeare-caesar.txt — The Tragedy of Julius...,0.020250,bryant-stories.txt — Stories by William Cullen...,0.230861
8,9,austen-sense.txt — Sense and Sensibility,0.019481,melville-moby_dick.txt — Moby Dick,0.229501
9,10,austen-emma.txt — Emma,0.019415,chesterton-thursday.txt — The Man Who Was Thur...,0.229203


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.052267,carroll-alice.txt — Alice's Adventures in Wond...,0.998604
1,2,carroll-alice.txt — Alice's Adventures in Wond...,0.033166,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.981176
2,3,shakespeare-macbeth.txt — The Tragedy of Macbeth,0.033053,bryant-stories.txt — Stories by William Cullen...,0.963445
3,4,blake-poems.txt — Poems of William Blake,0.024691,bible-kjv.txt — The King James Bible,0.961577
4,5,bryant-stories.txt — Stories by William Cullen...,0.023504,melville-moby_dick.txt — Moby Dick,0.954779
5,6,bible-kjv.txt — The King James Bible,0.020392,edgeworth-parents.txt — The Parent's Assistant,0.937438
6,7,shakespeare-caesar.txt — The Tragedy of Julius...,0.015432,milton-paradise.txt — Paradise Lost,0.931789
7,8,milton-paradise.txt — Paradise Lost,0.013213,chesterton-brown.txt — The Wisdom of Father Brown,0.894028
8,9,whitman-leaves.txt — Leaves of Grass,0.010148,chesterton-ball.txt — The Ball and the Cross,0.889671
9,10,edgeworth-parents.txt — The Parent's Assistant,0.008275,blake-poems.txt — Poems of William Blake,0.851488


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,chesterton-brown.txt — The Wisdom of Father Brown,0.024556,chesterton-brown.txt — The Wisdom of Father Brown,1.474927
1,2,blake-poems.txt — Poems of William Blake,0.023323,bible-kjv.txt — The King James Bible,1.441346
2,3,austen-persuasion.txt — Persuasion,0.016963,bryant-stories.txt — Stories by William Cullen...,1.426244
3,4,edgeworth-parents.txt — The Parent's Assistant,0.012693,austen-persuasion.txt — Persuasion,1.424482
4,5,austen-emma.txt — Emma,0.012250,austen-sense.txt — Sense and Sensibility,1.423008
5,6,bryant-stories.txt — Stories by William Cullen...,0.010732,austen-emma.txt — Emma,1.407071
6,7,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.010628,edgeworth-parents.txt — The Parent's Assistant,1.401308
7,8,austen-sense.txt — Sense and Sensibility,0.008108,chesterton-ball.txt — The Ball and the Cross,1.392694
8,9,bible-kjv.txt — The King James Bible,0.007296,chesterton-thursday.txt — The Man Who Was Thur...,1.343901
9,10,chesterton-ball.txt — The Ball and the Cross,0.006154,melville-moby_dick.txt — Moby Dick,1.340008


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,chesterton-ball.txt — The Ball and the Cross,0.008066,chesterton-brown.txt — The Wisdom of Father Brown,4.337437
1,2,whitman-leaves.txt — Leaves of Grass,0.007998,chesterton-ball.txt — The Ball and the Cross,4.334890
2,3,chesterton-brown.txt — The Wisdom of Father Brown,0.007978,whitman-leaves.txt — Leaves of Grass,3.632174
3,4,shakespeare-hamlet.txt — The Tragedy of Hamlet,0.004708,melville-moby_dick.txt — Moby Dick,2.332821
4,5,milton-paradise.txt — Paradise Lost,0.004468,shakespeare-hamlet.txt — The Tragedy of Hamlet,1.850363
5,6,shakespeare-macbeth.txt — The Tragedy of Macbeth,0.004022,chesterton-thursday.txt — The Man Who Was Thur...,1.553381
6,7,blake-poems.txt — Poems of William Blake,0.003757,milton-paradise.txt — Paradise Lost,0.984556
7,8,melville-moby_dick.txt — Moby Dick,0.003101,edgeworth-parents.txt — The Parent's Assistant,0.751587
8,9,chesterton-thursday.txt — The Man Who Was Thur...,0.002223,bible-kjv.txt — The King James Bible,0.470235
9,10,austen-persuasion.txt — Persuasion,0.002038,austen-emma.txt — Emma,0.057593


,consulta,tamaño TF-IDF,tamaño BM25,intersección top 10,primer TF-IDF,primer BM25
0,frecuente corta,10,10,8,chesterton-thursday.txt,bible-kjv.txt
1,frecuente media,10,10,7,blake-poems.txt,whitman-leaves.txt
2,raro corto,6,6,6,melville-moby_dick.txt,melville-moby_dick.txt
3,raro media,10,10,9,melville-moby_dick.txt,melville-moby_dick.txt
4,larga narrativa,10,10,9,chesterton-brown.txt,bible-kjv.txt
5,larga literaria,10,10,7,shakespeare-hamlet.txt,carroll-alice.txt
6,temática,10,10,8,chesterton-brown.txt,chesterton-brown.txt
7,mixta,10,10,7,chesterton-ball.txt,chesterton-brown.txt


,doc_id,|d|,término,"f(t,d)",df(t),aporte BM25,posición TF-IDF,posición BM25
0,whitman-leaves.txt,119331,ocean,36,9,1.474827,2,3
1,whitman-leaves.txt,119331,ship,79,12,0.907121,2,3
2,whitman-leaves.txt,119331,whale,5,6,1.896076,2,3
3,bryant-stories.txt,44790,ocean,4,9,1.312783,3,2
4,bryant-stories.txt,44790,ship,5,12,0.815710,3,2
5,bryant-stories.txt,44790,whale,16,6,2.268169,3,2


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [7]:
consulta_parametros = "raro media"
configuraciones = [("Base", 1.2, 0.75), ("Sin ajuste por longitud", 1.2, 0.0), ("Mayor efecto de las repeticiones", 2.0, 0.75)]
ids_referencia = [fila[0] for fila in ordenar_resultados(resultados[consulta_parametros]["bm25"])]
documentos_comparados = ids_referencia[:5]
filas_parametros = []
for nombre, k1, b in configuraciones:
    scores = puntuaciones_bm25(resultados[consulta_parametros]["tokens"], k1=k1, b=b)
    ranking = ordenar_resultados(scores)
    ids = [fila[0] for fila in ranking]
    for doc_id in documentos_comparados:
        i = ids.index(doc_id) + 1 if doc_id in ids else None
        filas_parametros.append({"Ejecución": nombre, "k1": k1, "b": b, "doc_id": doc_id,
                                 "posición": i, "score": scores[identificadores.index(doc_id)]})
tabla_parametros = pd.DataFrame(filas_parametros)
display(tabla_parametros.pivot(index="doc_id", columns="Ejecución", values="posición"))
display(tabla_parametros)


Ejecución,Base,Mayor efecto de las repeticiones,Sin ajuste por longitud
doc_id,,,
bryant-stories.txt,2,2,3
chesterton-ball.txt,5,5,4
melville-moby_dick.txt,1,1,1
shakespeare-hamlet.txt,4,4,5
whitman-leaves.txt,3,3,2


,Ejecución,k1,b,doc_id,posición,score
0,Base,1.2,0.75,melville-moby_dick.txt,1,4.763493
1,Base,1.2,0.75,bryant-stories.txt,2,4.396661
2,Base,1.2,0.75,whitman-leaves.txt,3,4.278024
3,Base,1.2,0.75,shakespeare-hamlet.txt,4,3.604763
4,Base,1.2,0.75,chesterton-ball.txt,5,3.568492
5,Sin ajuste por longitud,1.2,0.00,melville-moby_dick.txt,1,4.779189
6,Sin ajuste por longitud,1.2,0.00,bryant-stories.txt,3,4.111055
7,Sin ajuste por longitud,1.2,0.00,whitman-leaves.txt,2,4.286178
8,Sin ajuste por longitud,1.2,0.00,shakespeare-hamlet.txt,5,2.743749
9,Sin ajuste por longitud,1.2,0.00,chesterton-ball.txt,4,3.320917


### Interpretación

Cuando $b=0$, se elimina la penalización por longitud; por eso se favorece relativamente a los documentos largos.

Al aumentar $k_1$, las repeticiones conservan más peso antes de saturarse; por lo tanto, los términos con mayor frecuencia $f(t,d)$ cambian más su aporte.